In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from geopy.distance import geodesic
import math
import asyncio
from playwright.async_api import async_playwright
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
# Fixer le répertoire de travail sur la racine de ton projet
os.chdir('/Users/7maksacod/Documents/ISM/Master_2/Memoire_big_data_ia')

## SCRAPPING COIN AFRIQUE

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}

def extraire_details_annonce(url_annonce):
    """
    Ouvre la page d'une annonce spécifique et extrait toutes les données profondes.
    """
    try:
        response = requests.get(url_annonce, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            return None
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        donnees = {
            "url": url_annonce,
            "titre": None,
            "prix_fcfa": None,
            "adresse": None,
            "nb_pieces": None,
            "nb_salles_bain": None,
            "surface_m2": None,
            "description_complete": None,
            "images": []
        }
        
        # 1. Titre, Prix et Adresse
        titre_tag = soup.find('h1') or soup.find('h2')
        if titre_tag: 
            donnees["titre"] = titre_tag.text.strip()
        
        # Recherche du prix
        texte_complet = soup.get_text()
        prix_tag = soup.find(class_=lambda c: c and 'price' in c.lower())
        if prix_tag:
            prix_clean = ''.join(filter(str.isdigit, prix_tag.text))
            if prix_clean:
                donnees["prix_fcfa"] = int(prix_clean)
            
        # 2. Description textuelle complète (Mine d'or NLP)
        # On cherche le conteneur principal de la description
        desc_box = soup.find(class_=lambda c: c and ('description' in c.lower() or 'detail' in c.lower()))
        if desc_box:
            donnees["description_complete"] = desc_box.text.strip()
        else:
            # Fallback sur les paragraphes
            paragraphes = [p.text.strip() for p in soup.find_all('p') if len(p.text.strip()) > 30]
            if paragraphes:
                donnees["description_complete"] = " ".join(paragraphes)
            
        # 3. Extraction des URLs d'images (Haute Résolution pour le Deep Learning)
        images_tags = soup.find_all('img')
        for img in images_tags:
            src = img.get('src') or img.get('data-src')
            if src and ('coinafrique' in src or 'cloudinary' in src) and not 'logo' in src.lower():
                if src not in donnees["images"]:
                    donnees["images"].append(src)
                
        return donnees
        
    except Exception as e:
        print(f"⚠️ Erreur lors de l'extraction de {url_annonce} : {e}")
        return None


def scraper_coinafrique_deep(nb_pages=2):
    """
    Parcourt le catalogue, extrait les liens uniques, puis lance l'exploration profonde.
    """
    base_url = "https://sn.coinafrique.com/categorie/appartements?page="
    toutes_les_annonces = []
    
    for page in range(1, nb_pages + 1):
        print(f"\n================ PAGE {page}/{nb_pages} ================")
        url_catalogue = base_url + str(page)
        
        try:
            response = requests.get(url_catalogue, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Récupération de tous les liens vers des annonces
            liens_bruts = [a['href'] for a in soup.find_all('a', href=True) if '/annonce/' in a['href']]
            
            # Déduplication et construction des URLs complètes
            liens_uniques = []
            for l in liens_bruts:
                lien_full = "https://sn.coinafrique.com" + l if l.startswith('/') else l
                if lien_full not in liens_uniques:
                    liens_uniques.append(lien_full)
            
            print(f"🎯 {len(liens_uniques)} annonces uniques identifiées sur la page {page}.")
            print("Début du Deep Scraping (pause de 1.5s à 3s entre chaque annonce)...")
            
            for idx, lien in enumerate(liens_uniques, 1):
                details = extraire_details_annonce(lien)
                if details and details["titre"]:
                    toutes_les_annonces.append(details)
                    titre_court = (details['titre'][:35] + '...') if len(details['titre']) > 35 else details['titre']
                    print(f"  [{idx}/{len(liens_uniques)}] ✅ Extrait : {titre_court} | {details['prix_fcfa']} FCFA")
                
                # Temps de pause aléatoire pour protéger le serveur et éviter les blocages
                time.sleep(random.uniform(1.5, 3.0))
                
        except Exception as e:
            print(f"❌ Erreur sur la page catalogue {page} : {e}")
            
    # Sauvegarde dans un fichier CSV structuré
    df = pd.DataFrame(toutes_les_annonces)
    fichier_sortie = "coinafrique_deep_dataset.csv"
    df.to_csv(fichier_sortie, index=False, sep=";")
    print(f"\n🚀 MISSION ACCOMPLIE ! {len(df)} annonces complètes extraites et sauvegardées dans '{fichier_sortie}'.")
    return df

# Lancement du test sur 2 pages
if __name__ == "__main__":
    df_coinafrique = scraper_coinafrique_deep(nb_pages=30)


================ PAGE 1/30 ================
🎯 90 annonces uniques identifiées sur la page 1.
Début du Deep Scraping (pause de 1.5s à 3s entre chaque annonce)...
  [1/90] ✅ Extrait : Location studio  à Fann-mermoz | None FCFA
  [2/90] ✅ Extrait : Location appartements 3 pièces - Po... | None FCFA
  [3/90] ✅ Extrait : Location appartements 4 pièces - Me... | None FCFA
  [4/90] ✅ Extrait : Location appartement 3 pièces - Sac... | None FCFA
  [5/90] ✅ Extrait : Location appartement 4 pièces - Zac... | None FCFA
  [6/90] ✅ Extrait : Location appartements 2 pièces - Ba... | None FCFA
  [7/90] ✅ Extrait : Location appartements 2 pièces - Po... | None FCFA
  [8/90] ✅ Extrait : Location appartements 1 pièces - po... | None FCFA
  [9/90] ✅ Extrait : Location appartements 3 pièces - Ng... | None FCFA
  [10/90] ✅ Extrait : Location studio F2 séparé à Dakar p... | None FCFA
  [11/90] ✅ Extrait : Location appartement 3pièces  - Mam... | None FCFA
  [12/90] ✅ Extrait : Location appartements 2 pièces

## SCRAPPING EXPAT DAKAR

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}

def extraire_details_expatdakar(url_annonce):
    """
    Extrait le contenu brut d'une annonce Expat-Dakar pour notre Data Lake.
    """
    try:
        response = requests.get(url_annonce, headers=HEADERS, timeout=12)
        if response.status_code != 200:
            return None
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        donnees = {
            "source": "Expat-Dakar",
            "url": url_annonce,
            "titre": None,
            "prix_fcfa": None,
            "description_complete": None,
            "images": []
        }
        
        # 1. Extraction du Titre
        titre_tag = soup.find('h1')
        if titre_tag:
            donnees["titre"] = titre_tag.text.strip()
            
        # 2. Extraction globale du contenu texte (Zone Bronze)
        # On capture tout le bloc principal (souvent balise 'main' ou div contenant l'annonce)
        bloc_principal = soup.find('main') or soup.find('div', class_=lambda c: c and 'listing' in c.lower())
        if bloc_principal:
            # On récupère tout le texte, le script Regex se chargera de trier plus tard
            donnees["description_complete"] = bloc_principal.text.strip()
        else:
            donnees["description_complete"] = soup.get_text().strip()
            
        # 3. Extraction des URLs d'images HD
        images_tags = soup.find_all('img')
        for img in images_tags:
            src = img.get('src') or img.get('data-src')
            # Filtre pour éviter les logos et icônes du site
            if src and ('listing' in src or 'uploads' in src) and not 'logo' in src.lower() and not 'avatar' in src.lower():
                if src not in donnees["images"]:
                    donnees["images"].append(src)
                    
        return donnees
        
    except Exception as e:
        print(f"⚠️ Erreur sur {url_annonce} : {e}")
        return None


def scraper_expatdakar_deep(nb_pages=2):
    """
    Crawler ciblé sur les pages Appartements à Louer d'Expat-Dakar.
    """
    base_url = "https://www.expat-dakar.com/appartements-a-louer?page="
    toutes_les_annonces = []
    
    for page in range(1, nb_pages + 1):
        print(f"\n================ EXPAT-DAKAR : PAGE {page}/{nb_pages} ================")
        url_catalogue = base_url + str(page)
        
        try:
            response = requests.get(url_catalogue, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Filtre strict : on ne garde que les liens d'annonces individuelles (singulier)
            liens_bruts = [a['href'] for a in soup.find_all('a', href=True) if '/annonce/' in a['href']]
            
            # Déduplication et nettoyage du lien
            liens_uniques = []
            for l in liens_bruts:
                # Nettoyage des paramètres d'URL (les "?..." à la fin) pour éviter les doublons
                lien_propre = l.split('?')[0] 
                if not lien_propre.startswith('http'):
                    lien_propre = "https://www.expat-dakar.com" + lien_propre
                
                if lien_propre not in liens_uniques:
                    liens_uniques.append(lien_propre)
            
            print(f"🎯 {len(liens_uniques)} annonces uniques prêtes à être extraites.")
            
            for idx, lien in enumerate(liens_uniques, 1):
                details = extraire_details_expatdakar(lien)
                if details and details["titre"]:
                    toutes_les_annonces.append(details)
                    titre_court = (details['titre'][:40] + '...') if len(details['titre']) > 40 else details['titre']
                    print(f"  [{idx}/{len(liens_uniques)}] ✅ {titre_court}")
                
                time.sleep(random.uniform(1.5, 3.5))
                
        except Exception as e:
            print(f"❌ Erreur sur la page {page} : {e}")
            
    df = pd.DataFrame(toutes_les_annonces)
    fichier_sortie = "expatdakar_bronze_dataset.csv"
    df.to_csv(fichier_sortie, index=False, sep=";")
    print(f"\n🚀 MISSION EXPAT-DAKAR ACCOMPLIE ! {len(df)} annonces sauvegardées dans '{fichier_sortie}'.")
    return df

# Lancer le test sur 2 pages
if __name__ == "__main__":
    df_expat = scraper_expatdakar_deep(nb_pages=105)


================ EXPAT-DAKAR : PAGE 1/105 ================
🎯 10 annonces uniques prêtes à être extraites.
  [1/10] ✅ Villa Standing Cité Touba Renaissance / ...
  [2/10] ✅ Appartement F4 standing à louer à Sacre-...
  [3/10] ✅ Appartement F5 en ville avenue djily mba...
  [4/10] ✅ Appartement neuf  aux Almadies zone de r...
  [5/10] ✅ Appartement F4 + dépendance neuf à louer...
  [6/10] ✅ Appartement  avec grande terrasse aux al...
  [7/10] ✅ Appartement à louer à mermoz
  [8/10] ✅ À LOUER – F3 DE STANDING À SACRÉ-CŒUR 1
  [9/10] ✅ Appartement de type F4 à louer aux almad...
  [10/10] ✅ APPART A LOUER GUEDIAWAY

================ EXPAT-DAKAR : PAGE 2/105 ================
🎯 10 annonces uniques prêtes à être extraites.
  [1/10] ✅ Appartement à louer fenêtre Mermoz vue s...
  [2/10] ✅ Appartement à louer habitation ou bureau...
  [3/10] ✅ Ref_2587_Des appartements F4 à louer aux...
  [4/10] ✅ Ref_2937_GL Appartement F5, à louer au v...
  [5/10] ✅ Un appartement à louer aux Almadies
  [6/1

In [3]:
import pandas as pd
import numpy as np
import re

print("📥 Traitement de l'export WhatsApp...")

# 1. Lecture du fichier texte
chemin_fichier = "data/raw/_chat 4.txt" # Ton fichier export WhatsApp

with open(chemin_fichier, "r", encoding="utf-8") as file:
    lignes = file.readlines()

donnees = []
message_actuel = ""
date_actuelle, auteur_actuel = "", ""

# Regex adaptée pour le format [DD/MM/YYYY HH:MM:SS] Auteur: Message
pattern_debut = re.compile(r"^\[(\d{2}/\d{2}/\d{4})\s\d{2}:\d{2}:\d{2}\]\s(.*?):\s(.*)")

for ligne in lignes:
    match = pattern_debut.match(ligne)
    if match:
        if message_actuel:
            donnees.append([date_actuelle, auteur_actuel, message_actuel.strip()])
        date_actuelle, auteur_actuel, message_debut = match.groups()
        message_actuel = message_debut + " "
    else:
        message_actuel += ligne.strip() + " "

if message_actuel:
    donnees.append([date_actuelle, auteur_actuel, message_actuel.strip()])

df_wa = pd.DataFrame(donnees, columns=['date', 'auteur', 'description_complete'])

# 2. Filtrage Amélioré (Smart Regex)
mots_cles_location = ['louer', 'location', 'studio', 'appartement', 'chambre', 'salon', 'villa']
mots_cles_bruit = ['vendre', 'vente', 'terrain', 'canapé', 'iphone', 'voiture', 'ordinateur', 'drap']

def est_annonce_immo_valide(texte):
    texte = str(texte).lower()
    
    # Doit parler d'immobilier
    contient_location = any(mot in texte for mot in mots_cles_location)
    
    # Doit contenir un indicateur de prix large (cfa, fcfa, f, fr, mille, mil, k, ou un nombre à 5 chiffres ex: 80.000)
    contient_monnaie = re.search(r'(cfa|fcfa|\df|\d\s?f\b|fr\b|mille|mil\b|k\b|\d{2}\.?\d{3})', texte) is not None
    
    # Rejette les ventes et autres objets
    contient_bruit = any(mot in texte for mot in mots_cles_bruit)
    
    # Rejette les pures demandes courtes ("Bonjour je cherche un studio...")
    est_demande = re.search(r'^(bonjour\s)?(je\s)?cherche', texte) is not None and len(texte) < 100
    
    return contient_location and contient_monnaie and not contient_bruit and not est_demande

df_wa_clean = df_wa[df_wa['description_complete'].apply(est_annonce_immo_valide)].copy()

# 3. Formatage pour la Fusion Silver
df_wa_clean['source'] = 'WhatsApp'
df_wa_clean['titre'] = 'Annonce WhatsApp'
df_wa_clean['url'] = 'whatsapp://groupe_immo_dkr'
df_wa_clean['prix_fcfa'] = np.nan
df_wa_clean['nb_pieces'] = np.nan
df_wa_clean['nb_salles_bain'] = np.nan

colonnes_finales = ['source', 'url', 'titre', 'prix_fcfa', 'nb_pieces', 'nb_salles_bain', 'description_complete']
df_wa_final = df_wa_clean[colonnes_finales]

fichier_sortie = "data/raw/whatsapp_bronze_dataset.csv"
df_wa_final.to_csv(fichier_sortie, index=False, sep=";")

print(f"✅ Extraction réussie : {len(df_wa_final)} annonces immobilières WhatsApp valides récupérées !")

📥 Traitement de l'export WhatsApp...
✅ Extraction réussie : 377 annonces immobilières WhatsApp valides récupérées !


## SCRAPPING FACEBOOK MARKET

In [14]:
import pandas as pd
import os

# 1. Chemin vers votre dataset d'origine multisources
INPUT_PATH = "data/processed/dataset_dakar_master_multisources.csv"
OUTPUT_PATH = "data/raw/annonces_facebook_marketplace.csv"

if os.path.exists(INPUT_PATH):
    # Lecture dynamique du fichier CSV avec détection de séparateur
    df = pd.read_csv(INPUT_PATH, sep=None, engine='python')
    print(f"📊 Dataset d'origine chargé : {df.shape[0]} lignes.")

    # 2. Filtrage strict des annonces provenant de Facebook Marketplace
    df_fb = df[df['source'] == 'Facebook Marketplace'].copy()
    
    # Nettoyage optionnel des colonnes invisibles (ex: \ufefftitre -> titre)
    df_fb.columns = df_fb.columns.str.replace('\ufeff', '').str.strip()

    print(f"🎯 Nombre d'annonces Facebook Marketplace identifiées : {len(df_fb)}")

    # 3. Exportation vers un fichier CSV dédié
    df_fb.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f"✅ Fichier exporté avec succès : '{OUTPUT_PATH}'")

else:
    print(f"⚠️ Fichier introuvable à l'emplacement : {INPUT_PATH}")

📊 Dataset d'origine chargé : 1017 lignes.
🎯 Nombre d'annonces Facebook Marketplace identifiées : 201
✅ Fichier exporté avec succès : 'data/raw/annonces_facebook_marketplace.csv'
